# Data Mining — Activity I: Data Visualization and Dashboard Deployment

**Universidad de la Costa**  
**Professor:** José Escorcia-Gutierrez, Ph.D.  

**Team:**
- Diego Navarro Gómez (Group 18690)
- Juan Félix (Group 18038)
- Dinelis García (Group 18038)
- Kimberly Ochoa (Group 19027)

---

## Goal
Load the university student dataset, explore its structure, document each column, and create visualizations that summarize retention rate trends, student satisfaction scores, and the comparison between Spring and Fall terms.

## Step 1 — Import Libraries

In [ ]:
# pandas is used to load and work with the dataset (rows, columns, groupby, etc.)
import pandas as pd

# matplotlib is the base library for creating charts
import matplotlib.pyplot as plt

# seaborn is built on top of matplotlib and gives us nicer default styles
import seaborn as sns

# We set a clean style so all charts look consistent
sns.set_style("whitegrid")

print("Libraries imported successfully.")

## Step 2 — Load the Dataset

In [ ]:
# We load the CSV file into a DataFrame
# If you are running this in Google Colab, upload the file first using the Files panel
df = pd.read_csv("university_student_data.csv")

# We print the first 5 rows to confirm the file loaded correctly
df.head()

## Step 3 — Explore the Dataset Structure

In [ ]:
# shape tells us how many rows and columns the dataset has
print("Dataset shape (rows, columns):", df.shape)

# dtypes tells us the data type of each column (int, float, object, etc.)
print("\nData types:")
print(df.dtypes)

In [ ]:
# describe() gives us basic statistics: min, max, mean, and standard deviation
# This helps us understand the range and distribution of numeric columns
df.describe()

In [ ]:
# We check for missing values — if any column has nulls we would need to handle them
print("Missing values per column:")
print(df.isnull().sum())

# We also check the unique terms to confirm only Spring and Fall are present
print("\nUnique terms:", df["Term"].unique())
print("Years covered:", sorted(df["Year"].unique()))

## Step 4 — Column Documentation

Here we explain what each column means in the context of this study:

| Column | Type | Description |
|--------|------|-------------|
| **Year** | Integer | The academic year when the data was recorded (2015–2024). |
| **Term** | String | The semester — **Spring** (first half of the year) or **Fall** (second half). |
| **Applications** | Integer | Total number of students who applied to the university that term. |
| **Admitted** | Integer | Number of applicants who were accepted by the university. |
| **Enrolled** | Integer | Number of admitted students who actually confirmed their enrollment. |
| **Retention Rate (%)** | Float | Percentage of students who stayed enrolled and did not drop out. A higher value means the university is keeping its students. |
| **Student Satisfaction (%)** | Float | Average satisfaction score reported by students. Higher means students are happier with the university. |
| **Engineering Enrolled** | Integer | Number of enrolled students who chose the Engineering department. |
| **Business Enrolled** | Integer | Number of enrolled students who chose the Business department. |
| **Arts Enrolled** | Integer | Number of enrolled students who chose the Arts department. |
| **Science Enrolled** | Integer | Number of enrolled students who chose the Science department. |

> **Note:** Each row represents one year + one term combination. Since there are two terms per year (Spring and Fall) and data from 2015 to 2024, the dataset has 20 rows total.

## Step 5 — Visualization 1: Retention Rate Trends Over Time

We use a **line chart** here because we want to see how the retention rate changes over the years. A line chart is the best option when we have a continuous time variable on the x-axis.

In [ ]:
# We group by Year and compute the mean retention rate
# This averages Spring and Fall into a single yearly value so the line is smoother
retention_by_year = df.groupby("Year")["Retention Rate (%)"].mean().reset_index()

plt.figure(figsize=(10, 5))

# plot() creates a line chart — we add markers so each year is clearly marked
plt.plot(
    retention_by_year["Year"],
    retention_by_year["Retention Rate (%)"],
    marker="o",
    color="steelblue",
    linewidth=2,
    label="Retention Rate"
)

# We annotate each point so the reader can see the exact value without guessing
for _, row in retention_by_year.iterrows():
    plt.text(row["Year"], row["Retention Rate (%)"] + 0.15, f"{row['Retention Rate (%)']:.0f}%",
             ha="center", fontsize=9, color="steelblue")

plt.title("Retention Rate Trend Over Time (2015–2024)", fontsize=14)
plt.xlabel("Year")
plt.ylabel("Retention Rate (%)")
plt.ylim(82, 93)  # We narrow the y-axis so small changes are visible
plt.xticks(retention_by_year["Year"])
plt.legend()
plt.tight_layout()
plt.show()

print("\nRetention Rate by Year:")
print(retention_by_year.to_string(index=False))

## Step 6 — Visualization 2: Student Satisfaction Scores by Year

We use **sns.lineplot()** as required by the activity. Seaborn handles the grouping internally and gives a nice-looking chart. This is useful for showing trends between a time variable and a numeric value.

In [ ]:
# We group by Year and take the average satisfaction to get one point per year
satisfaction_by_year = df.groupby("Year")["Student Satisfaction (%)"].mean().reset_index()

plt.figure(figsize=(10, 5))

# sns.lineplot() is used here because the activity explicitly asks for it
# It also handles confidence intervals automatically if there is more than one value per x
sns.lineplot(
    data=satisfaction_by_year,
    x="Year",
    y="Student Satisfaction (%)",
    marker="o",
    color="coral",
    linewidth=2,
    label="Satisfaction Score"
)

# We annotate each data point for easy reading
for _, row in satisfaction_by_year.iterrows():
    plt.text(row["Year"], row["Student Satisfaction (%)"] + 0.2,
             f"{row['Student Satisfaction (%)']:.0f}%",
             ha="center", fontsize=9, color="coral")

plt.title("Student Satisfaction Scores by Year (2015–2024)", fontsize=14)
plt.xlabel("Year")
plt.ylabel("Student Satisfaction (%)")
plt.ylim(74, 92)
plt.xticks(satisfaction_by_year["Year"])
plt.legend()
plt.tight_layout()
plt.show()

print("\nStudent Satisfaction by Year:")
print(satisfaction_by_year.to_string(index=False))

## Step 7 — Visualization 3: Spring vs Fall Comparison

We use a **bar chart** to compare the two terms. A bar chart is ideal when we want to compare discrete categories (Spring and Fall) on a common metric.

In [ ]:
# We group by Term and calculate the average of the key numeric columns
# This gives us one row per term so we can compare them directly
term_comparison = df.groupby("Term")[["Applications", "Admitted", "Enrolled",
                                       "Retention Rate (%)", "Student Satisfaction (%)"]].mean()

print("Average values per term:")
print(term_comparison.round(2))

In [ ]:
# We create a grouped bar chart so the user can compare multiple metrics at once
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Spring vs Fall Comparison", fontsize=15)

# We define the metrics and colors to keep the chart clean
metrics = ["Enrolled", "Retention Rate (%)", "Student Satisfaction (%)"]
colors = ["#DD8452", "#4C72B0"]  # Orange for Spring, blue for Fall
terms = term_comparison.index.tolist()  # ["Fall", "Spring"]

for i, metric in enumerate(metrics):
    # We use bar() to create one bar per term for each metric
    axes[i].bar(terms, term_comparison[metric], color=colors, width=0.4)
    axes[i].set_title(metric)
    axes[i].set_ylabel(metric)
    axes[i].set_xlabel("Term")
    # We add value labels on top of each bar for clarity
    for bar in axes[i].patches:
        axes[i].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f"{bar.get_height():.1f}",
            ha="center", va="bottom", fontsize=11
        )

plt.tight_layout()
plt.show()

## Step 8 — Summary of Findings

Based on the three visualizations above, we can observe:

1. **Retention Rate** has grown steadily from **85% in 2015** to **90% in 2024**. This means the university has improved its ability to keep students enrolled over the years. The only small dip was in **2020**, likely due to the pandemic disruptions.

2. **Student Satisfaction** also increased consistently from **78% in 2015** to **88% in 2024**. This positive trend suggests that improvements in services, infrastructure, or teaching quality are working.

3. **Spring vs Fall comparison** shows that both terms are practically identical in enrollment numbers, retention rate, and satisfaction. This means the university operates consistently across both semesters — there is no "better" or "worse" term.

### Actionable Insight
> Since retention and satisfaction both improved together, the university should continue investigating **what changed between 2015 and 2024** — for example, new student support programs or improved course quality — and apply those same improvements to the **Science department**, where enrollment has been declining in recent years.